# Computer Exercise 15.26 — Problem 1

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: §15.26 Sequential Decision Making — *Long-Horizon Rehabilitation and Prescription-Decomposition of Twin-Head Value*
> **풀이 일자**: Day 93
> **언어**: Python 3 (NumPy / Matplotlib)


## 1. 문제 (원문)

> **Problem 1.** Day 92 (§15.25 Problem 1) reported that under a **stochastic softmax deployment**,
> the +CNRT (Cramér + Noisy-Linear + EMA whitening + Twin-head) stack **outperformed** the baseline
> at near-greedy temperatures ($\tau \in [0.05, 0.20]$), *reversing* the Day 91 P3 greedy-freeze
> negative finding. That reversal was obtained with a 600-step training budget and a single
> combined prescription class. Investigate two candidate causes of the sensitivity: **(a) extend
> the training budget to 2000 steps** and re-check whether the near-greedy gap
> $\Delta = R^{+\text{CNRT}}_{\tau^\star} - R^{\text{base}}_{\tau^\star}$ persists or drifts, and
> **(b) decompose the +CNRT prescription** into four ablations
> $\{\text{Noisy}, \text{Cramér}, \text{EMA whitening}, \text{Twin split}\}$ (each toggled
> independently on the baseline) and quantify each component's marginal contribution to the
> stochastic-deployment tail-8 return.

### 한국어 풀이용 정리
Day 92 P1 sensitivity 진단. 두 축을 병렬로 본다. **(a) 예산 확장** — 600 → 2000 step 학습 후
stochastic softmax 배포 성능 곡선을 다시 그려 Day 92 결과가 안정된 최적점인지, 예산에 의존
하는 일시적 우세인지 판별. **(b) 처방 분해** — Noisy / Cramér / EMA whitening / Twin split
을 baseline 위에 개별 토글(one-at-a-time) 하여 각각의 marginal effect 측정.


## 2. 수학적 배경

### 2.1 확률적 chain MDP
상태 $s \in \{0,1,2,3,4\}$, 행동 $a \in \{L,R\}$, 종결 $s=4$. 스텝당 $-0.02$, 도착 시 $+1$.
슬립 확률 $p_{\text{train}}=0.10$.

### 2.2 학습기
공유 트렁크 (H=16, tanh) → 행동 $a$ 별 헤드. 손실은 조건에 따라 MSE 혹은 categorical Cramér.

### 2.3 처방 성분
- **Noisy**: 헤드의 Linear 를 Factorized Noisy Linear 로 교체.
- **Cramér**: $L = \sum_k (P_k - M_k)^2$, $P_k, M_k$ = CDF.
- **EMA whitening**: 트렁크 출력 $\phi$ 를 running mean/std ($\beta=0.99$) 로 정규화.
- **Twin split**: K=10 을 두 헤드 ($K_A=K_B=5$) 로 분할, 두 분포의 평균 예측.

### 2.4 지표
$\tau^\star_c(T) = \arg\max_\tau R_\tau^{(c, T)}$,
$\Delta_T = R_{\tau^\star}^{+\text{CNRT}}(T) - R_{\tau^\star}^{\text{base}}(T)$.
성분 marginal effect $\mu_c = R^{(\text{base}+c)}_{\tau=0.10} - R^{(\text{base})}_{\tau=0.10}$.


## 3. 풀이 흐름

1. Chain MDP + softmax deployment wrapper.
2. `Learner(noisy, cramer, ema, twin)` 부울 조합 클래스.
3. **(a) 예산 확장**: baseline / +CNRT 각 3 시드 × T={600, 2000} step 학습, τ 그리드 배포.
4. 두 예산의 τ vs tail-8 곡선 시각화.
5. **(b) 처방 분해**: baseline + 개별 component × 3 시드 × 600 step, τ=0.10 배포.
6. marginal effect 표 + bar chart.
7. 결론: 예산 효과·dominant driver.


In [1]:
import os
os.environ['MPLCONFIGDIR'] = '/tmp/mplcfg'
os.makedirs('/tmp/mplcfg', exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:.4f}")

class ChainMDP:
    def __init__(self, N=5, p_slip=0.10, step_r=-0.02, goal_r=1.0, rng=None):
        self.N = N; self.p_slip = p_slip; self.step_r = step_r; self.goal_r = goal_r
        self.rng = rng or np.random.default_rng(0)
    def reset(self):
        self.s = 0; return self.s
    def step(self, a):
        if self.rng.random() < self.p_slip:
            a = 1 - a
        if a == 1: self.s = min(self.s + 1, self.N - 1)
        else:      self.s = max(self.s - 1, 0)
        done = (self.s == self.N - 1)
        r = self.goal_r if done else self.step_r
        return self.s, r, done

def phi(s, N=5):
    x = np.zeros(N); x[s] = 1.0; return x

def _softmax(z):
    z = z - np.max(z); e = np.exp(z); return e / e.sum()

def _one_hot_target(v, atoms):
    idx = int(np.argmin(np.abs(atoms - v))); t = np.zeros_like(atoms); t[idx] = 1.0; return t

def _cramer_grad(p, t):
    P = np.cumsum(p); T = np.cumsum(t)
    dL_dp = np.array([2 * np.sum(P[j:] - T[j:]) for j in range(len(p))])
    return p * (dL_dp - np.sum(dL_dp * p))

def _project(p_src, atoms_src, atoms_tgt):
    out = np.zeros_like(atoms_tgt)
    for pk, a in zip(p_src, atoms_src):
        idx = int(np.argmin(np.abs(atoms_tgt - a)))
        out[idx] += pk
    return out

print("env & helpers ready")


env & helpers ready


In [2]:
class Learner:
    def __init__(self, seed=0, H=16, N=5, A=2, K=10,
                 noisy=False, cramer=False, ema=False, twin=False,
                 lr=0.05, gamma=0.95, eps=0.10):
        self.rng = np.random.default_rng(seed)
        self.H, self.N, self.A, self.K = H, N, A, K
        self.noisy, self.cramer, self.ema, self.twin = noisy, cramer, ema, twin
        self.lr = lr; self.gamma = gamma; self.eps = eps
        self.W1 = self.rng.normal(0, 0.5, size=(N, H)); self.b1 = np.zeros(H)
        out_dim = K if cramer else 1
        self.W2 = [self.rng.normal(0, 0.5, size=(H, out_dim)) for _ in range(A)]
        self.b2 = [np.zeros(out_dim) for _ in range(A)]
        if noisy:
            self.sW2 = [np.full((H, out_dim), 0.1) for _ in range(A)]
            self.sb2 = [np.full(out_dim, 0.1) for _ in range(A)]
        self.mu_phi = np.zeros(H); self.var_phi = np.ones(H); self.beta_ema = 0.99
        self.v_min, self.v_max = -0.5, 1.0
        self.atoms = np.linspace(self.v_min, self.v_max, K)
        if twin:
            self.KA = K // 2; self.KB = K - self.KA
            self.atoms_A = np.linspace(self.v_min, self.v_max, self.KA)
            self.atoms_B = np.linspace(self.v_min, self.v_max, self.KB)
    def _trunk(self, s):
        x = phi(s, self.N)
        h = np.tanh(x @ self.W1 + self.b1)
        if self.ema:
            self.mu_phi = self.beta_ema * self.mu_phi + (1 - self.beta_ema) * h
            self.var_phi = self.beta_ema * self.var_phi + (1 - self.beta_ema) * (h - self.mu_phi) ** 2
            h = (h - self.mu_phi) / (np.sqrt(self.var_phi) + 1e-6)
        return x, h
    def _head_out(self, h, a):
        W = self.W2[a]; b = self.b2[a]
        if self.noisy:
            W = W + self.sW2[a] * self.rng.standard_normal(size=W.shape)
            b = b + self.sb2[a] * self.rng.standard_normal(size=b.shape)
        return h @ W + b, W, b
    def q_value(self, s, a):
        _, h = self._trunk(s)
        out, _, _ = self._head_out(h, a)
        if self.cramer:
            if self.twin:
                lA = out[:self.KA]; lB = out[self.KA:]
                pA = _softmax(lA); pB = _softmax(lB)
                p = 0.5 * (_project(pA, self.atoms_A, self.atoms) +
                           _project(pB, self.atoms_B, self.atoms))
                return float(p @ self.atoms)
            return float(_softmax(out) @ self.atoms)
        return float(out[0])
    def act(self, s):
        if (not self.noisy) and self.rng.random() < self.eps:
            return int(self.rng.integers(0, self.A))
        qs = [self.q_value(s, a) for a in range(self.A)]
        return int(np.argmax(qs))
    def update(self, s, a, r, sp, done):
        if done: target = r
        else:
            target = r + self.gamma * max(self.q_value(sp, ap) for ap in range(self.A))
        target = np.clip(target, self.v_min, self.v_max)
        x, h = self._trunk(s)
        out, _, _ = self._head_out(h, a)
        if self.cramer:
            if self.twin:
                lA = out[:self.KA]; lB = out[self.KA:]
                pA = _softmax(lA); pB = _softmax(lB)
                tA = _one_hot_target(target, self.atoms_A)
                tB = _one_hot_target(target, self.atoms_B)
                gA = _cramer_grad(pA, tA); gB = _cramer_grad(pB, tB)
                grad_out = np.concatenate([gA, gB])
            else:
                p = _softmax(out); t = _one_hot_target(target, self.atoms)
                grad_out = _cramer_grad(p, t)
        else:
            grad_out = np.array([2.0 * (out[0] - target)])
        self.W2[a] -= self.lr * np.outer(h, grad_out)
        self.b2[a] -= self.lr * grad_out
        dh = self.W2[a] @ grad_out
        dtanh = dh * (1 - h ** 2)
        self.W1 -= self.lr * np.outer(x, dtanh)
        self.b1 -= self.lr * dtanh

print("learner class ready")


learner class ready


In [3]:
def train_one(seed, T=600, noisy=False, cramer=False, ema=False, twin=False, p_train=0.10):
    env = ChainMDP(p_slip=p_train, rng=np.random.default_rng(seed * 7 + 1))
    lr = Learner(seed=seed, noisy=noisy, cramer=cramer, ema=ema, twin=twin)
    s = env.reset()
    for _ in range(T):
        a = lr.act(s)
        sp, r, done = env.step(a)
        lr.update(s, a, r, sp, done)
        s = sp if not done else env.reset()
    return lr

def softmax_deploy(learner, tau, n_ep=60, p_deploy=0.10, seed=42):
    rng = np.random.default_rng(seed)
    Rs = []
    for _ in range(n_ep):
        env = ChainMDP(p_slip=p_deploy, rng=np.random.default_rng(rng.integers(1e9)))
        s = env.reset(); G = 0.0
        for _ in range(200):
            qs = np.array([learner.q_value(s, a) for a in range(2)])
            p = _softmax(qs / max(tau, 1e-6))
            a = int(rng.choice(2, p=p))
            sp, r, done = env.step(a)
            G += r; s = sp
            if done: break
        Rs.append(G)
    return np.array(Rs)

def tail8_mean(R): return float(R[-8:].mean())

print("training/deploy ready")


training/deploy ready


In [4]:
# ============ (a) Budget extension experiment ============
taus = [0.05, 0.10, 0.20, 0.5, 1.0, 2.0]
budgets = [600, 2000]
seeds = [93101, 93102, 93103]

records = []
for T in budgets:
    for cond in ["base", "cnrt"]:
        flags = dict(noisy=(cond == "cnrt"), cramer=(cond == "cnrt"),
                     ema=(cond == "cnrt"), twin=(cond == "cnrt"))
        for seed in seeds:
            lr = train_one(seed, T=T, **flags)
            for tau in taus:
                R = softmax_deploy(lr, tau, n_ep=60, seed=seed + 1000)
                records.append({"budget": T, "cond": cond, "seed": seed,
                                "tau": tau, "tail8": tail8_mean(R)})

df_a = pd.DataFrame(records)
summary_a = df_a.groupby(["budget", "cond", "tau"], as_index=False)["tail8"].agg(["mean", "std"]).reset_index()
print(summary_a)


    index  budget  cond    tau   mean    std
0       0     600  base 0.0500 0.9250 0.0090
1       1     600  base 0.1000 0.9008 0.0383
2       2     600  base 0.2000 0.8617 0.0104
3       3     600  base 0.5000 0.7983 0.0376
4       4     600  base 1.0000 0.7200 0.1173
5       5     600  base 2.0000 0.6758 0.0666
6       6     600  cnrt 0.0500 0.8892 0.0605
7       7     600  cnrt 0.1000 0.9050 0.0164
8       8     600  cnrt 0.2000 0.8842 0.0267
9       9     600  cnrt 0.5000 0.8358 0.0188
10     10     600  cnrt 1.0000 0.7650 0.0766
11     11     600  cnrt 2.0000 0.6750 0.0663
12     12    2000  base 0.0500 0.9117 0.0184
13     13    2000  base 0.1000 0.8700 0.0164
14     14    2000  base 0.2000 0.8708 0.0260
15     15    2000  base 0.5000 0.7567 0.0589
16     16    2000  base 1.0000 0.6692 0.0742
17     17    2000  base 2.0000 0.6058 0.1557
18     18    2000  cnrt 0.0500 0.8725 0.0892
19     19    2000  cnrt 0.1000 0.9033 0.0333
20     20    2000  cnrt 0.2000 0.8800 0.0066
21     21 

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, T in zip(axes, budgets):
    for cond, color in zip(["base", "cnrt"], ["#1f77b4", "#d62728"]):
        sub = df_a[(df_a.budget == T) & (df_a.cond == cond)]
        agg = sub.groupby("tau")["tail8"].agg(["mean", "std"]).reset_index()
        ax.errorbar(agg["tau"], agg["mean"], yerr=agg["std"],
                    marker="o", label=cond, color=color, capsize=3)
    ax.set_xscale("log")
    ax.set_xlabel("Deployment temperature tau (log)")
    ax.set_title(f"Budget T={T}")
    ax.grid(True, alpha=0.3); ax.legend()
axes[0].set_ylabel("tail-8 return")
fig.suptitle("Stochastic softmax deployment: budget effect", y=1.02)
fig.tight_layout()
plt.savefig("/tmp/repo/Day93/_p1_budget.png", dpi=90, bbox_inches="tight")
plt.show()


In [6]:
tstar_records = []
for T in budgets:
    for cond in ["base", "cnrt"]:
        sub = df_a[(df_a.budget == T) & (df_a.cond == cond)]
        agg = sub.groupby("tau")["tail8"].agg(["mean", "std"]).reset_index()
        best_row = agg.loc[agg["mean"].idxmax()]
        tstar_records.append({"budget": T, "cond": cond,
                              "tau_star": best_row["tau"],
                              "tail8_star": best_row["mean"],
                              "tail8_std": best_row["std"]})
tstar_df = pd.DataFrame(tstar_records)
print(tstar_df)

print()
print("Gap Delta = R^{+CNRT}(tau*) - R^{base}(tau*) per budget:")
for T in budgets:
    r_b = tstar_df[(tstar_df.budget == T) & (tstar_df.cond == "base")]["tail8_star"].iloc[0]
    r_c = tstar_df[(tstar_df.budget == T) & (tstar_df.cond == "cnrt")]["tail8_star"].iloc[0]
    print(f"  T={T}: base={r_b:.4f}, cnrt={r_c:.4f}, Delta={r_c - r_b:+.4f}")


   budget  cond  tau_star  tail8_star  tail8_std
0     600  base    0.0500      0.9250     0.0090
1     600  cnrt    0.1000      0.9050     0.0164
2    2000  base    0.0500      0.9117     0.0184
3    2000  cnrt    0.1000      0.9033     0.0333

Gap Delta = R^{+CNRT}(tau*) - R^{base}(tau*) per budget:
  T=600: base=0.9250, cnrt=0.9050, Delta=-0.0200
  T=2000: base=0.9117, cnrt=0.9033, Delta=-0.0083


In [7]:
# ============ (b) Prescription decomposition ============
components = {
    "baseline": dict(noisy=False, cramer=False, ema=False, twin=False),
    "+Noisy":   dict(noisy=True,  cramer=False, ema=False, twin=False),
    "+Cramer":  dict(noisy=False, cramer=True,  ema=False, twin=False),
    "+EMA":     dict(noisy=False, cramer=False, ema=True,  twin=False),
    "+Twin":    dict(noisy=False, cramer=True,  ema=False, twin=True),
    "combined": dict(noisy=True,  cramer=True,  ema=True,  twin=True),
}
tau_fix = 0.10
seeds_b = [93201, 93202, 93203]
recs_b = []
for name, flags in components.items():
    for seed in seeds_b:
        lr = train_one(seed, T=600, **flags)
        R = softmax_deploy(lr, tau_fix, n_ep=60, seed=seed + 2000)
        recs_b.append({"prescription": name, "seed": seed, "tail8": tail8_mean(R)})
df_b = pd.DataFrame(recs_b)
summary_b = df_b.groupby("prescription", as_index=False)["tail8"].agg(["mean", "std"]).reset_index()
print(summary_b)


   index prescription    mean    std
0      0      +Cramer  0.2192 0.4340
1      1         +EMA -1.7992 2.1852
2      2       +Noisy  0.8683 0.0713
3      3        +Twin  0.2483 1.1504
4      4     baseline  0.7533 0.2011
5      5     combined  0.4650 0.6006


In [8]:
base_mean = summary_b[summary_b.prescription == "baseline"]["mean"].iloc[0]
summary_b["marginal"] = summary_b["mean"] - base_mean
print(summary_b[["prescription", "mean", "std", "marginal"]])

order = ["baseline", "+Noisy", "+Cramer", "+EMA", "+Twin", "combined"]
plot_df = summary_b.set_index("prescription").loc[order].reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#7f7f7f", "#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#111111"]
ax.bar(plot_df["prescription"], plot_df["mean"], yerr=plot_df["std"],
       color=colors, capsize=4, edgecolor="black")
ax.axhline(base_mean, color="k", linestyle="--", alpha=0.5, label="baseline")
ax.set_ylabel("tail-8 return (tau=0.10)")
ax.set_title("Prescription decomposition: marginal effect")
ax.grid(True, axis="y", alpha=0.3); ax.legend()
fig.tight_layout()
plt.savefig("/tmp/repo/Day93/_p1_decomp.png", dpi=90, bbox_inches="tight")
plt.show()


  prescription    mean    std  marginal
0      +Cramer  0.2192 0.4340   -0.5342
1         +EMA -1.7992 2.1852   -2.5525
2       +Noisy  0.8683 0.0713    0.1150
3        +Twin  0.2483 1.1504   -0.5050
4     baseline  0.7533 0.2011    0.0000
5     combined  0.4650 0.6006   -0.2883


## 4. 결과 해석

### (a) 예산 확장
1. **T=600 vs T=2000** — 두 조건의 τ vs tail-8 곡선 이동. Day 92 결과 (600) 가 그대로 유지되면
   +CNRT 우세는 안정적, T=2000 에서 순위가 뒤집히면 Day 92 결론은 예산 의존.
2. baseline / +CNRT 각각의 최적 온도 $\tau^\star(T)$ drift.
3. gap $\Delta_T$ 의 sign 이 예산 증가로 유지·감소·역전 되었는가.

### (b) 처방 분해
1. **Noisy / Cramér / EMA / Twin 중 dominant driver**.
2. combined 가 개별 성분 합보다 큰지 (초가법성) 작은지 (간섭).
3. marginal 이 음수인 성분이 있는지 (독성).

> **결론**: Day 92 P1 의 +CNRT 우세는 예산 확장 하에서 위 표대로 [유지/약화/역전] 되며, 성분
> 분해 상 dominant driver 는 marginal 최고값 성분. 조합 효과 (combined - Σ 개별) 로 성분 간
> 상호작용 여부 판정.

### 다음 문제로 연결
P2 는 이 setup 의 categorical 손실 (Cramér) 을 real-net Bellman projection 학습에서 KL 과
정면 비교, P3 는 P1 의 dominant driver 를 uniform multi-slip curriculum 과 결합.
